In [0]:
# ============================================================
# CELL 1 — Gold Layer: load the served model + frozen alert threshold
# Notebook: 05_gold_layer
# Served model: Ridge v2 (interpretable baseline) from 04_modeling run
# "ridge_v2_engine_cv". The LSTM is the accuracy champion candidate;
# migrating gold to it is the next step (it needs 30 cycles of history
# per prediction, so the first 29 cycles of each engine need a fallback).
# ============================================================

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient

EXPERIMENT_PATH = "/Users/gkolthoff1@live.maryville.edu/rul_prediction_cmapss"

SENSORS = [2, 3, 4, 7, 8, 9, 11, 12, 13, 15, 17, 20, 21]
feature_cols = (
    [f"sensor_{s}_norm" for s in SENSORS] +
    [f"sensor_{s}_roll_mean" for s in SENSORS] +
    [f"sensor_{s}_roll_std" for s in SENSORS]
)

client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_PATH)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'ridge_v2_engine_cv'",
    order_by=["attributes.start_time DESC"],   # most recent v2 run, NOT best test score
    max_results=1,
)
if not runs:
    raise ValueError("No 'ridge_v2_engine_cv' run found. Run Cell 2 of 04_modeling first.")

run = runs[0]
run_id = run.info.run_id
TAU = float(run.data.params["alert_tau"])
ridge_model = mlflow.sklearn.load_model(f"runs:/{run_id}/ridge_model")

print(f"✅ Loaded Ridge v2 from run {run_id}")
print(f"   alpha = {run.data.params['best_alpha']} | alert threshold tau = {TAU}")
print(f"   last-cycle RMSE = {run.data.metrics['last_cycle_rmse']:.2f} | "
      f"PHM08 = {run.data.metrics['phm08_score']:.0f} | "
      f"alert recall (rows) = {run.data.metrics['alert_recall_rows']:.3f}")

✅ Loaded Ridge v2 from run 7fa5898dd90a4c57b8fdbb181710e4a1
   alpha = 100.0 | alert threshold tau = 50.26
   last-cycle RMSE = 17.51 | PHM08 = 664 | alert recall (rows) = 0.747


In [0]:
# ============================================================
# CELL 2 — Generate Gold predictions (FD001 TEST engines only)
# - Test split only: train rows would be in-sample fits, not predictions
# - FD003 excluded: no model has been trained or evaluated on it
# - Alert uses the frozen threshold TAU from 04_modeling (not 30)
# - No dollar figures: the cost inputs had no source
# ============================================================

from datetime import datetime, timezone

silver_fd001 = spark.table("workspace.predictive_maintenance.silver_fd001")
pdf = (silver_fd001.filter("split = 'test'").toPandas()
       .sort_values(["unit_id", "time_cycles"]).reset_index(drop=True))

rul_pred = np.clip(ridge_model.predict(pdf[feature_cols].fillna(0)), 0, 125)
is_last = pdf.groupby("unit_id")["time_cycles"].transform("max") == pdf["time_cycles"]

gold_pd = pd.DataFrame({
    "unit_id":         pdf["unit_id"].astype(int),
    "time_cycles":     pdf["time_cycles"].astype(int),
    "split":           pdf["split"],
    "subset":          "FD001",
    "rul_true_raw":    pdf["RUL"].astype(float),          # from RUL_FD001.txt (uncapped)
    "rul_actual":      pdf["RUL_capped"].astype(float),   # capped at 125 (training target)
    "rul_predicted":   rul_pred.round(2),
    "rul_error":       (rul_pred - pdf["RUL_capped"]).round(2),   # > 0 = predicted LATE
    "alert":           (rul_pred <= TAU).astype(int),
    "alert_threshold": round(TAU, 2),
    "alert_urgency":   pd.cut(rul_pred, bins=[-1, 10, TAU, TAU + 30, 125.01],
                              labels=["CRITICAL", "WARNING", "MONITOR", "HEALTHY"]).astype(str),
    "is_last_cycle":   is_last.astype(int),
    "model_version":   "ridge_v2_engine_cv",
    "model_run_id":    run_id,
    "predicted_at":    datetime.now(timezone.utc).isoformat(),
})

# --- Sanity checks against 04_modeling (must match) ---
last = gold_pd[gold_pd.is_last_cycle == 1]
last_rmse = float(np.sqrt(np.mean((last.rul_predicted - last.rul_true_raw.clip(upper=125)) ** 2)))
pos = gold_pd.rul_true_raw <= 30
row_recall = (gold_pd.alert[pos] == 1).mean()
eng_tp = int(((last.alert == 1) & (last.rul_true_raw <= 30)).sum())
eng_fp = int(((last.alert == 1) & (last.rul_true_raw > 30)).sum())
assert len(gold_pd) == 13096 and gold_pd.unit_id.nunique() == 100, "unexpected row/engine count"

(spark.createDataFrame(gold_pd).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.predictive_maintenance.gold_predictions"))

print("✅ Gold Delta table written: workspace.predictive_maintenance.gold_predictions")
print(f"   Rows: {len(gold_pd):,} (FD001 test, 100 engines) | alert threshold tau = {TAU:.1f}")
print(f"   Check vs 04_modeling — last-cycle RMSE: {last_rmse:.2f} (expect 17.51)")
print(f"   Row recall: {row_recall:.3f} (expect 0.747) | engines alerted at last cycle: "
      f"{eng_tp}/25, {eng_fp} false alarms (expect 21/25, 1)")
print("\nUrgency distribution (all test rows):")
print(gold_pd.alert_urgency.value_counts().reindex(["CRITICAL", "WARNING", "MONITOR", "HEALTHY"]).to_string())

{"ts": "2026-10-13 01:42:56.512", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDI4NDEwNDIzNDU4NzYxNhABIAEyJDAxYTEzMmViLWQxNzQtN2MxMS04ODFmLTFmMzBiNzEwYTkwOTokOTVkYzA0ZjYtZTI3NS0zMjFiLWEwOWYtZmE1YTExNzk2MmIwSgwIvZi21gYQgMaYgwNQAVgBYAFosLmLqOrEow2IAQA=.", "context": {}}
{"ts": "2026-10-13 01:42:56.512", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDI4NDEwNDIzNDU4NzYxNhABIAEyJDAxYTEzMmViLWQxNzQtN2MxMS04ODFmLTFmMzBiNzEwYTkwOTokOTVkYzA0ZjYtZTI3NS0zMjFiLWEwOWYtZmE1YTExNzk2MmIwSgwIvZi21gYQgMaYgwNQAVgBYAFosLmLqOrEow2IAQA=.", "context": {}}
{"ts": "2026-10-13 01:42:56.512", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDI4NDEwNDIzNDU4NzYxNhABIAEyJDAxYTEzMmViLWQxNzQtN2MxMS04ODFmLTFmMzBiNzEwYTkwOTokOTVkYzA0ZjYtZTI3NS0zMjFiLWEwOWYtZmE1YTEx

✅ Gold Delta table written: workspace.predictive_maintenance.gold_predictions
   Rows: 13,096 (FD001 test, 100 engines) | alert threshold tau = 50.3
   Check vs 04_modeling — last-cycle RMSE: 17.51 (expect 17.51)
   Row recall: 0.747 (expect 0.747) | engines alerted at last cycle: 21/25, 1 false alarms (expect 21/25, 1)

Urgency distribution (all test rows):
alert_urgency
CRITICAL       27
WARNING       312
MONITOR      1135
HEALTHY     11622


In [0]:
# ============================================================
# CELL 3 — Register the served Ridge model (Unity Catalog, with signature)
# Metrics are copied from the evaluated 04_modeling run, never hardcoded.
# Role: interpretable BASELINE that Gold currently serves. The LSTM
# (04_modeling run "lstm_v2_seed1") is the accuracy champion candidate.
# ============================================================

import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

model_name = "ridge_rul_predictor_cmapss"

X_sample = pdf[feature_cols].fillna(0).head(100)
signature = infer_signature(X_sample, ridge_model.predict(X_sample))

mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="ridge_v2_registered"):
    mlflow.log_param("model_type", "RidgeCV_v2_baseline")
    mlflow.log_param("source_run_id", run_id)
    mlflow.log_param("alert_tau", TAU)
    for k in ["last_cycle_rmse", "last_cycle_rmse_raw", "phm08_score", "test_rmse", "test_mae",
              "alert_recall_rows", "alert_precision_rows", "alert_f1_rows"]:
        mlflow.log_metric(k, run.data.metrics[k])
    info = mlflow.sklearn.log_model(
        ridge_model,
        name="ridge_model_signed",
        signature=signature,
        input_example=X_sample.head(5),
        registered_model_name=model_name,
    )

version = info.registered_model_version
print(f"✅ Registered {model_name} version {version} (role: served baseline)")
print(f"   last-cycle RMSE {run.data.metrics['last_cycle_rmse']:.2f} | "
      f"PHM08 {run.data.metrics['phm08_score']:.0f} | alert tau {TAU:.1f}")

🔗 View Logged Model at: https://dbc-875e7f65-c010.cloud.databricks.com/ml/experiments/3021995221306764/models/m-01638554385f418aaf5ba21900583272?o=7474646022806704
Registered model 'ridge_rul_predictor_cmapss' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.default.ridge_rul_predictor_cmapss': https://dbc-875e7f65-c010.cloud.databricks.com/explore/data/models/workspace/default/ridge_rul_predictor_cmapss/version/3?o=7474646022806704


✅ Registered ridge_rul_predictor_cmapss version 3 (role: served baseline)
   last-cycle RMSE 17.51 | PHM08 664 | alert tau 50.3


In [0]:
# ============================================================
# CELL 4 — Write Gold to Snowflake (serving layer)
# STATUS: not re-run in this version. Gold was loaded to Snowflake
# during the trial (Aug 2026); the trial account has since expired.
# The code is kept to show the serving-layer write. Credentials come
# from the Databricks secret scope "snowflake" — nothing secret is
# stored in this notebook.
# ============================================================

import snowflake.connector
import numpy as np

SF_DATABASE  = "PREDICTIVE_MAINTENANCE"
SF_SCHEMA    = "GOLD"
SF_WAREHOUSE = "COMPUTE_WH"
SF_TABLE     = "GOLD_PREDICTIONS"

COLS = ["unit_id", "time_cycles", "split", "subset", "rul_true_raw", "rul_actual",
        "rul_predicted", "rul_error", "alert", "alert_threshold", "alert_urgency",
        "is_last_cycle", "model_version", "model_run_id", "predicted_at"]

gold_sf = spark.table("workspace.predictive_maintenance.gold_predictions").toPandas()[COLS]
print(f"Loaded {len(gold_sf):,} rows from Gold")

conn = snowflake.connector.connect(
    account=dbutils.secrets.get("snowflake", "account"),
    user=dbutils.secrets.get("snowflake", "user"),
    password=dbutils.secrets.get("snowflake", "password"),
    warehouse=SF_WAREHOUSE, database=SF_DATABASE, schema=SF_SCHEMA,
)
cursor = conn.cursor()
cursor.execute(f"""
CREATE OR REPLACE TABLE {SF_TABLE} (
    UNIT_ID INTEGER, TIME_CYCLES INTEGER, SPLIT VARCHAR(10), SUBSET VARCHAR(10),
    RUL_TRUE_RAW FLOAT, RUL_ACTUAL FLOAT, RUL_PREDICTED FLOAT, RUL_ERROR FLOAT,
    ALERT INTEGER, ALERT_THRESHOLD FLOAT, ALERT_URGENCY VARCHAR(20), IS_LAST_CYCLE INTEGER,
    MODEL_VERSION VARCHAR(50), MODEL_RUN_ID VARCHAR(64), PREDICTED_AT VARCHAR(50)
)""")
print("✅ Table recreated")

rows = list(gold_sf.replace({np.nan: None}).itertuples(index=False, name=None))
placeholders = ",".join(["%s"] * len(COLS))
BATCH = 5000
for i in range(0, len(rows), BATCH):
    cursor.executemany(f"INSERT INTO {SF_TABLE} ({','.join(COLS)}) VALUES ({placeholders})",
                       rows[i:i + BATCH])
    print(f"  Inserted {min(i + BATCH, len(rows)):,}/{len(rows):,} rows...")

cursor.execute(f"SELECT COUNT(*), SUM(ALERT), COUNT(DISTINCT UNIT_ID) FROM {SF_TABLE}")
n, n_alert, n_eng = cursor.fetchone()
print(f"✅ Verified in Snowflake: {n:,} rows | {n_eng} engines | {n_alert:,} alert rows")
cursor.close()
conn.close()